# DoubleIntegrator — Stage 2 — Sparse Dictionary Policy Training (DPC)

Train and inspect the sparse SD-DPC policy.


## 1. Set up the system and load the stage config

Load dependencies and configuration.


In [ ]:
"""Boilerplate: make the in-repo `sdpc` package importable and resolve this system."""
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "sdpc").is_dir() and (candidate / "case_studies").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the SD-DPC repository")

REPO_ROOT = find_repo_root()
CASE_DIR = REPO_ROOT / "case_studies" / "DoubleIntegrator"
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
from sdpc.config import load_config, seed_everything
from sdpc.registry import make_system

SYSTEM = "double_integrator"
device = torch.device("cpu")
system = make_system(SYSTEM, device=device)
CONFIGS = CASE_DIR / "configs"
RESULTS = CASE_DIR / "results"

print(f"System        : {SYSTEM}")
print(f"State dim nx  : {system.nx}")
print(f"Control dim nu: {system.nu}")
print(f"Sample time ts: {system.ts}")
print(f"Input bounds  : [{system.umin}, {system.umax}]")
print(f"State bounds  : [{system.xmin}, {system.xmax}]")
print(f"Discrete model: {system.is_discrete}")


In [ ]:
cfg = load_config(CONFIGS / 'policy.yaml')
seed_everything(int(cfg.get('seed', 0)))

print('Resolved policy-training config:')
for k, v in cfg.items():
    if not k.startswith('_'):
        print(f'  {k}: {v}')

## 2. Load the identified dynamics model from Stage 1

Restore the required checkpoints.


In [ ]:
from sdpc.io import find_dynamics_checkpoint
from sdpc.sindy import load_model

dynamics_path = find_dynamics_checkpoint(RESULTS, cfg)
print('Using dynamics checkpoint:', dynamics_path)

sindy = load_model(dynamics_path, device=device)
print('\nLoaded dynamics model:')
sindy.pretty_print()

## 3. Build the sparse policy library and model

Define the system and problem.


In [ ]:
from sdpc.sindy import CompiledFunctionLibrary, SINDyVectorized

policy_lib_cfg = system.policy_library_cfg()
print('Policy library config:', policy_lib_cfg)

plib = CompiledFunctionLibrary(**policy_lib_cfg)
policy = SINDyVectorized(
    library=plib, n_out=system.nu,
    policy_name=[f'u{i}' for i in range(system.nu)],
    device=device, seed=cfg.get('policy_seed', 0),
)
init_scale = cfg.get('init_scale')
if init_scale is not None:
    for i in range(system.nu):
        policy.Xi[i].data.mul_(init_scale)

print(f'\nPolicy library has {plib.shape[0]} candidate terms:')
print(plib.function_names)
print('\nInitial (untrained) policy:')
policy.pretty_print()

## 4. Generate policy-training data

Prepare the required training data.


In [ ]:
train_loader, dev_loader = system.make_policy_data(cfg, device)

batch = next(iter(train_loader))
print('Train batch keys:', list(batch.keys()))
print('xn shape (batch, 1, nx):', tuple(batch['xn'].shape))
print('r  shape (batch, horizon+1, nref):', tuple(batch['r'].shape))

## 5. Train the sparse policy (differentiable predictive control)

Optimize the controller parameters.


In [ ]:
from sdpc.training import make_debias_config, train_policy
from sdpc.io import CustomLogger

logger = CustomLogger(args=None, savedir=str(RESULTS / 'logs' / 'policy_nb'),
                      verbosity=cfg['verbosity'],
                      stdout=cfg.get('logger_metrics', ['train_loss', 'dev_loss']))
train_policy(system, sindy, policy, train_loader, dev_loader, cfg, device,
             logger=logger, action_scale=cfg.get('action_scale', 1.0))

if cfg.get('debias', {}).get('enabled', False):
    support_before_debias = [list(indices) for indices in policy.active_idx]
    debias_cfg = make_debias_config(cfg)
    train_policy(
        system, sindy, policy, train_loader, dev_loader,
        debias_cfg, device, logger=logger,
        action_scale=cfg.get('action_scale', 1.0),
    )
    assert [list(indices) for indices in policy.active_idx] == support_before_debias
    print(f"Completed {debias_cfg['epochs']} fixed-support policy debias epochs")

## 6. Inspect the learned policy

Inspect policy performance.


In [ ]:
print('Learned sparse policy:')
policy.pretty_print()
print()
print('Active terms per output:', [len(idx) for idx in policy.active_idx])
print('Total active terms      :', sum(len(idx) for idx in policy.active_idx))

## 7. Visualize the active coefficients

Inspect the surviving coefficients.


In [ ]:
from sdpc.plotting import plot_coef_heatmap
import matplotlib.pyplot as plt

plot_coef_heatmap(policy, title=f'{SYSTEM}: active policy coefficients')
plt.show()

## 8. Sanity-check a nominal closed-loop rollout

Visualize the closed-loop response.


In [ ]:
from sdpc.eval import make_test_data, rollout_closed_loop

nominal_plant = system.discrete_step(sindy)
eval_seed = cfg.get('eval_seed', 12)
eval_margin = cfg.get('eval_margin', 0.6)
generator = torch.Generator(device='cpu').manual_seed(eval_seed)
sample_scale = 1.0 - 2.0 * eval_margin
init_center = torch.tensor(cfg['init_center'], dtype=torch.float32)
ref_center = torch.tensor(cfg['ref_center'], dtype=torch.float32)
x0 = (init_center + cfg['init_side'] * sample_scale * (torch.rand(system.nx, generator=generator) - 0.5)).tolist()
ref = [-2.5,3.3]
data = make_test_data(system.nx, cfg["nsteps"], x0, ref, device=device)

res = rollout_closed_loop(policy, nominal_plant, data,
                          umin=system.umin, umax=system.umax,
                          action_scale=cfg.get('action_scale', 1.0))
print('evaluation seed:', eval_seed)
print('x0:', data['xn'][0, 0].tolist())
print('reference:', data['r'][0, 0].tolist())
print('final state:', res['x_traj'][0, -1].tolist())

### Obstacle-avoidance trajectory

Visualize the closed-loop response.


In [ ]:
from sdpc.plotting import plot_trajectories_with_ellipse

e = cfg['ellipse']
plot_trajectories_with_ellipse(
    [
        {
            'traj': res['x_traj'],
            'label': 'learned sparse policy',
            'color': 'green',
        }
    ],
    r=data['r'],
    p=e['p'], b=e['b'], c=e['c'], d=e['d'], theta=e.get('theta', 0.0),
    xmin=system.xmin, xmax=system.xmax,
    title=f'{SYSTEM}: learned-policy obstacle avoidance',
)
plt.show()

## 9. Save the policy (optional)

Prefer `scripts/DoubleIntegrator/train_sd_dpc.py` for a tracked run; to save directly from this notebook, uncomment below.

In [ ]:
from sdpc.io import new_policy_run_dir, snapshot_config
from sdpc.sindy import save_model
run_dir = new_policy_run_dir(RESULTS, 'sparse')
save_model(policy, run_dir / 'saved_models' / 'policy_sparse.pt')
snapshot_config(run_dir, cfg)
print('Saved to', run_dir)